# 2장 06. 모델 평가 기초 실습

이 notebook은 모델 평가를 처음 보는 사람도 따라갈 수 있게, **데이터가 예측 결과와 평가 지표로 바뀌는 과정**을 한 단계씩 확인합니다.


### 따라하기 안내: 모델 평가 흐름

이 notebook은 한 셀에서 여러 결과를 한꺼번에 보여주지 않고, 중간 결과를 나누어 확인합니다.

1. 평가 데이터가 어떤 파일에서 왔는지 봅니다.
2. label이 지표 계산에 맞게 정리되는지 봅니다.
3. feature가 score 계산에 들어갈 수 있는 상태인지 봅니다.
4. score가 threshold를 지나 prediction으로 바뀌는 과정을 봅니다.
5. prediction과 label을 비교해 FP/FN과 metric을 읽습니다.
6. validation 데이터 조건이 달라졌을 때 metric이 어떻게 움직이는지 확인합니다.

출력 표를 볼 때는 “이 숫자가 어느 단계에서 나온 값인가?”를 계속 확인하세요. score, prediction, label, metric을 섞어 읽으면 해석이 틀어집니다.


## 0. scikit-learn 평가를 처음 보는 사람을 위한 기초 예제

2-4에서는 모델 평가를 다룹니다. 본 데이터에 들어가기 전에 scikit-learn에서 자주 보는 흐름을 작은 예제로 먼저 확인합니다.

| scikit-learn 표현 | 쉬운 뜻 | 이번 실습에서의 의미 |
| --- | --- | --- |
| `fit()` | 모델이 데이터를 보고 기준을 배우는 단계 | 학습 데이터로 모델을 만듭니다. |
| `predict_proba()` | 각 class일 가능성을 점수로 출력 | `high_risk` score를 만듭니다. |
| `predict()` | 최종 class 예측 | `high_risk` 또는 `low_risk`를 냅니다. |
| `confusion_matrix()` | 정답과 예측을 비교한 표 | TP, FP, FN, TN을 봅니다. |
| `precision_score()` | 예측한 positive 중 맞은 비율 | 오탐이 많으면 낮아집니다. |
| `recall_score()` | 실제 positive 중 찾아낸 비율 | 미탐이 많으면 낮아집니다. |

이 기초 예제는 모델 성능을 좋게 만드는 방법을 배우는 구간이 아닙니다. **score가 prediction이 되고, prediction이 metric으로 바뀌는 과정**을 이해하는 구간입니다.


### 0-1. 작은 학습 데이터 만들기

아래 예제는 scikit-learn API 흐름을 보기 위한 작은 데이터입니다. 실제 모델 품질을 확인하기에는 너무 작지만, `fit -> predict_proba -> predict -> metric` 순서를 보기에는 충분합니다.


In [ ]:
import pandas as pd

sklearn_basic_dataframe = pd.DataFrame(
    {
        "heart_rate": [72, 88, 96, 110, 76, 102],
        "oxygen_saturation": [98, 96, 94, 91, 99, 93],
        "label": ["low_risk", "low_risk", "high_risk", "high_risk", "low_risk", "high_risk"],
    }
)

display(sklearn_basic_dataframe)


**출력 확인**

입력 feature는 `heart_rate`, `oxygen_saturation` 두 개이고, 정답 label은 `high_risk` 또는 `low_risk`입니다.

scikit-learn 모델은 보통 feature 표 `X`와 정답 `y`를 나누어 받습니다.


### 0-2. scikit-learn API 흐름 실행하기

이 셀은 scikit-learn이 설치된 환경에서는 실제 `DecisionTreeClassifier`를 사용합니다. 설치되어 있지 않은 환경에서는 미리 정한 score로 같은 해석 흐름을 보여 줍니다.


In [ ]:
sklearn_basic_features = ["heart_rate", "oxygen_saturation"]
sklearn_basic_positive_label = "high_risk"

try:
    from sklearn.tree import DecisionTreeClassifier

    sklearn_basic_available = True
    sklearn_basic_model = DecisionTreeClassifier(max_depth=2, random_state=7)
    sklearn_basic_model.fit(
        sklearn_basic_dataframe.loc[:, sklearn_basic_features],
        sklearn_basic_dataframe["label"],
    )
    sklearn_basic_classes = list(sklearn_basic_model.classes_)
    sklearn_basic_positive_index = sklearn_basic_classes.index(sklearn_basic_positive_label)
    sklearn_basic_scores = sklearn_basic_model.predict_proba(
        sklearn_basic_dataframe.loc[:, sklearn_basic_features]
    )[:, sklearn_basic_positive_index]
except Exception as sklearn_basic_error:
    sklearn_basic_available = False
    sklearn_basic_model = None
    sklearn_basic_classes = ["high_risk", "low_risk"]
    sklearn_basic_scores = [0.05, 0.25, 0.72, 0.91, 0.18, 0.84]

sklearn_basic_threshold = 0.50
sklearn_basic_result = sklearn_basic_dataframe.assign(
    high_risk_score=sklearn_basic_scores,
    prediction=lambda table: table["high_risk_score"].ge(sklearn_basic_threshold).map(
        {True: "high_risk", False: "low_risk"}
    ),
    correct=lambda table: table["label"].eq(table["prediction"]),
)

sklearn_basic_status = pd.DataFrame(
    [
        {
            "item": "sklearn_available",
            "value": sklearn_basic_available,
            "meaning": "True이면 실제 scikit-learn 모델로 score를 만들었습니다.",
        },
        {"item": "threshold", "value": sklearn_basic_threshold, "meaning": "score를 prediction으로 바꾸는 기준입니다."},
    ]
)


**출력 확인: `sklearn_basic_status`**

먼저 실행 환경과 threshold를 확인합니다. scikit-learn이 없더라도 fallback score로 같은 해석 흐름을 따라갈 수 있습니다.


In [ ]:
display(sklearn_basic_status)

**출력 확인: `sklearn_basic_result`**

각 행의 score가 threshold를 지나 prediction으로 바뀐 결과를 봅니다. `correct`는 prediction이 label과 같은지 표시합니다.


In [ ]:
display(sklearn_basic_result)

**출력 확인**

`high_risk_score`는 아직 최종 예측이 아닙니다. `threshold=0.50`을 적용한 뒤에야 `prediction`이 됩니다.

- score가 0.50 이상이면 `high_risk`
- score가 0.50 미만이면 `low_risk`

본 실습에서도 이 구분이 중요합니다. score 분포와 prediction 분포는 같은 말이 아닙니다.


### 0-3. FP/FN과 precision/recall 계산하기

정답 `label`과 모델 `prediction`을 비교하면 오류 유형을 볼 수 있습니다. scikit-learn이 있으면 metric 함수를 쓰고, 없어도 같은 공식을 직접 계산할 수 있습니다.


In [ ]:
tp = int(((sklearn_basic_result["label"] == "high_risk") & (sklearn_basic_result["prediction"] == "high_risk")).sum())
fp = int(((sklearn_basic_result["label"] == "low_risk") & (sklearn_basic_result["prediction"] == "high_risk")).sum())
fn = int(((sklearn_basic_result["label"] == "high_risk") & (sklearn_basic_result["prediction"] == "low_risk")).sum())
tn = int(((sklearn_basic_result["label"] == "low_risk") & (sklearn_basic_result["prediction"] == "low_risk")).sum())

sklearn_basic_confusion = pd.DataFrame(
    [
        {"actual": "high_risk", "predicted_high_risk": tp, "predicted_low_risk": fn},
        {"actual": "low_risk", "predicted_high_risk": fp, "predicted_low_risk": tn},
    ]
)

sklearn_basic_metrics = pd.DataFrame(
    [
        {"metric": "precision", "value": tp / (tp + fp) if (tp + fp) else 0, "easy_meaning": "high_risk라고 예측한 것 중 실제 high_risk 비율"},
        {"metric": "recall", "value": tp / (tp + fn) if (tp + fn) else 0, "easy_meaning": "실제 high_risk 중 모델이 찾아낸 비율"},
    ]
).assign(value=lambda table: table["value"].round(4))


**출력 확인: `sklearn_basic_confusion`**

정답과 예측이 만나는 표입니다. `actual=high_risk` 행에서 `predicted_low_risk`가 있으면 미탐(FN)입니다.


In [ ]:
display(sklearn_basic_confusion)

**출력 확인: `sklearn_basic_metrics`**

confusion matrix에서 precision과 recall을 계산한 결과입니다. precision은 FP, recall은 FN의 영향을 받습니다.


In [ ]:
display(sklearn_basic_metrics)

**출력 확인**

confusion matrix는 모델 평가의 핵심입니다.

| 값 | 뜻 |
| --- | --- |
| TP | 실제 `high_risk`를 `high_risk`로 맞힘 |
| FP | 실제 `low_risk`를 `high_risk`로 잘못 예측 |
| FN | 실제 `high_risk`를 `low_risk`로 놓침 |
| TN | 실제 `low_risk`를 `low_risk`로 맞힘 |

이제 아래 본 실습에서는 같은 흐름을 실제 test 데이터와 validation 데이터에 적용합니다.


## 1. 실습 준비와 실행 범위

### 1-1. Lite와 로컬 실행 기준 고정

이 notebook에서는 먼저 어떤 실행 환경과 어떤 helper 기준으로 score와 metric을 계산하는지 고정하는 것입니다. JupyterLite에서는 browser-safe `ai_quality.lite` helper를 사용하고, 로컬에서는 저장소의 `packages/ai-quality/src`를 우선 사용합니다.

이 준비 셀은 `utils.py`를 통해 설치와 import만 담당합니다. 모델 평가의 핵심인 데이터 로딩, label 표준화, feature readiness, score 생성, threshold 적용, metric 계산은 뒤 셀에서 Pandas 코드로 드러냅니다.


### 따라하기 안내: 평가 환경 준비

**이 셀에서 하는 일**  
실습에 필요한 도구를 불러옵니다.

**확인할 점**  
패키지 이름, 기준 label, threshold처럼 뒤 셀에서 계속 사용할 기준값을 확인합니다.

**왜 중요한가**  
처음 기준이 잘못 잡히면 뒤의 모든 출력도 잘못 해석됩니다.


In [ ]:
from __future__ import annotations

import utils as ch02

prepared = await ch02.prepare_notebook()
pd = prepared.pandas
aiq_lite = prepared.aiq_lite

environment_summary = pd.DataFrame(
    [
        {"item": "package_module", "value": aiq_lite.__name__, "why_it_matters": "Lite와 로컬에서 같은 score/metric 기준을 제공합니다."},
        {"item": "positive_label", "value": aiq_lite.POSITIVE_LABEL, "why_it_matters": "recall과 precision을 계산할 관심 class입니다."},
        {"item": "negative_label", "value": aiq_lite.NEGATIVE_LABEL, "why_it_matters": "FP를 해석할 비교 class입니다."},
        {"item": "default_threshold", "value": aiq_lite.THRESHOLD, "why_it_matters": "score를 prediction으로 바꾸는 기본 기준입니다."},
    ]
)

display(environment_summary)


이 출력에서 확인할 핵심은 `package_module`과 기본 threshold입니다. `ai_quality.lite`는 Lite에서도 실행 가능한 기준값과 sample loader를 제공하지만, 실제 score 생성과 metric 계산은 아래 Pandas 셀에서 보여줍니다.

`high_risk`는 Positive class이고 `low_risk`는 Negative class입니다. FP는 실제 `low_risk`를 `high_risk`로 잘못 예측한 오탐이고, FN은 실제 `high_risk`를 `low_risk`로 놓친 미탐입니다.

## 2. 평가 데이터 brief와 변환 기준

### 2-1. test 원본 데이터 로딩과 row grain 확인

이 셀에서는 모델 평가가 어떤 원본 데이터에서 시작되는지 확인하는 것입니다. metric은 숫자 하나로 보이지만, 그 숫자는 특정 파일, 특정 행 수, 특정 class support, 특정 실행 범위에서만 의미가 있습니다.

`vital_signs_test.csv`의 한 row는 생체신호 기반 위험 알림 시스템의 평가용 sample 하나를 뜻합니다. 이 sample은 실제 의료 확인이 아니라 classification 예제의 입력과 정답 기준입니다.

이 단계에서는 아직 label을 표준화하지 않고 score도 만들지 않습니다. 먼저 source, sample scope, row count, column count, 원본 label 값을 고정해야 뒤에서 metric이 어떤 데이터에서 계산되었는지 설명할 수 있습니다.


### 따라하기 안내: test 데이터 확인

**이 셀에서 하는 일**  
최종 평가에 사용할 test 데이터를 확인합니다.

**확인할 점**  
row 수, column 수, 원본 label 값을 봅니다.

**왜 중요한가**  
metric은 이 데이터에서 계산됩니다. 데이터 범위를 모르면 성능 숫자를 해석할 수 없습니다.


In [ ]:
feature_columns = aiq_lite.FEATURE_COLUMNS
positive_label = aiq_lite.POSITIVE_LABEL
negative_label = aiq_lite.NEGATIVE_LABEL
operating_threshold = aiq_lite.THRESHOLD
TEST_DATA_PATH = "data/vital_signs_test.csv"

raw_test_dataframe, test_source = aiq_lite.load_csv_or_sample(
    TEST_DATA_PATH,
    aiq_lite.sample_vital_signs(),
    nrows=None,
)

execution_scope = "embedded_fallback_sample" if "embedded" in test_source else (
    "jupyterlite_sample" if len(raw_test_dataframe) <= 600 else "local_full_or_large_sample"
)

raw_provenance = pd.DataFrame(
    [
        {
            "dataset": "vital_signs_test",
            "dataset_role": "selected_test_evaluation_dataset",
            "path": TEST_DATA_PATH,
            "data_source": test_source,
            "execution_scope": execution_scope,
            "row_grain": "one classification sample for risk alert evaluation",
            "row_count": raw_test_dataframe.shape[0],
            "column_count": raw_test_dataframe.shape[1],
        }
    ]
)

raw_label_distribution = (
    raw_test_dataframe["label"]
    .value_counts(dropna=False)
    .rename_axis("raw_label")
    .reset_index(name="count")
    .assign(ratio_pct=lambda table: (table["count"] / len(raw_test_dataframe) * 100).round(2))
)

raw_preview_columns = [
    column
    for column in ["patient_id", "timestamp", *feature_columns, "label"]
    if column in raw_test_dataframe.columns
]

raw_data_brief = pd.DataFrame(
    [
        {"check": "source_fixed", "observed": test_source, "qa_use": "metric 계산의 입력 파일 또는 fallback sample 기준을 고정합니다."},
        {"check": "row_grain", "observed": "one classification sample", "qa_use": "metric 분모가 무엇을 세는지 설명합니다."},
        {"check": "raw_label_values", "observed": raw_label_distribution["raw_label"].astype(str).tolist(), "qa_use": "표준화 전 정답 기준 값을 확인합니다."},
        {"check": "score_status", "observed": "not_created_yet", "qa_use": "아직 모델 출력이 아니라 원본 평가 입력만 확인합니다."},
    ]
)


### 출력 확인: `raw_provenance`

**확인할 점**  
데이터나 artifact가 어디에서 왔는지 봅니다. 경로와 실행 범위를 먼저 확인합니다.

**왜 중요한가**  
같은 숫자라도 prepared artifact인지 local 재생성인지에 따라 보고서 표현이 달라집니다.

**다음 단계**  
이 범위를 기억한 상태로 다음 표를 읽습니다.


In [ ]:
display(raw_provenance)


### 출력 확인: `raw_data_brief`

**확인할 점**  
행 수, 컬럼 수, 데이터셋 이름을 봅니다.

**왜 중요한가**  
분석 대상이 기대와 다르면 뒤의 결측, label, metric 결과도 의미가 달라집니다.

**다음 단계**  
대상이 맞으면 preview와 label 분포로 넘어갑니다.


In [ ]:
display(raw_data_brief)


### 출력 확인: `raw_test_dataframe.loc[:, raw_preview_columns].head()`

**확인할 점**  
몇 개 행을 직접 보고 컬럼 이름과 값 모양이 예상과 맞는지 봅니다.

**왜 중요한가**  
Preview는 전체 확인이 아니라 데이터 모양 확인입니다.

**다음 단계**  
분포나 metric은 뒤의 요약 표에서 확인합니다.


In [ ]:
display(raw_test_dataframe.loc[:, raw_preview_columns].head())


### 출력 확인: `raw_label_distribution`

**확인할 점**  
`high_risk`와 `low_risk`의 개수와 비율을 봅니다.

**왜 중요한가**  
`high_risk`는 관심 class입니다. 이 수가 적으면 recall 같은 지표가 크게 흔들릴 수 있습니다.

**다음 단계**  
두 class가 모두 있는지 확인한 뒤 label gate로 넘어갑니다.


In [ ]:
display(raw_label_distribution)


이 출력에서 확인할 핵심은 아직 score나 prediction이 없다는 점입니다. 지금 보이는 것은 평가 입력의 원본 상태이며, 이 단계에서 row 수나 label 값이 기대와 다르면 metric 계산으로 넘어가면 안 됩니다.

`execution_scope`가 `jupyterlite_sample` 또는 `embedded_fallback_sample`이면 숫자는 브라우저 실습용 축약 근거입니다. 보고서에는 전체 로컬 데이터인지 Lite sample인지 반드시 구분해야 합니다.

### 2-2. label 표준화와 class support 확인

이 셀에서는 원본 label 값이 평가용 label 계약으로 안전하게 바뀌는지 확인하는 것입니다. label은 정답 기준이므로, score나 prediction을 만들기 전에 `high_risk`와 `low_risk`로 표준화되는 과정을 분리해서 봐야 합니다.

표준화는 원본 dataframe을 직접 덮어쓰지 않고 `copy()` 후 `assign()`으로 진행합니다. 이렇게 해야 원본 label과 표준화 label을 나란히 비교할 수 있고, 나중에 metric 문제가 생겼을 때 label 변환이 원인인지 추적할 수 있습니다.

이 단계의 통과 기준은 허용되지 않은 label이 없고, Positive class와 Negative class가 모두 충분히 존재하는 것입니다. class support가 한쪽으로 치우치면 Precision과 Recall을 해석할 때 제한 사항을 남겨야 합니다.


### 따라하기 안내: label 표준화

**이 셀에서 하는 일**  
label 표기를 `high_risk`, `low_risk`로 맞춥니다.

**확인할 점**  
`High Risk`, `high_risk`, `HIGH_RISK`처럼 같은 뜻의 값이 섞여 있지 않은지 봅니다.

**왜 중요한가**  
label은 모델 평가의 정답입니다. 표기가 섞이면 정답 기준이 흔들려 precision과 recall이 잘못 계산될 수 있습니다.


In [ ]:
test_dataframe = raw_test_dataframe.copy().assign(
    label=lambda table: table["label"].map(aiq_lite.normalize_label)
)

label_transition = (
    raw_test_dataframe.loc[:, ["label"]]
    .rename(columns={"label": "raw_label"})
    .assign(standardized_label=test_dataframe["label"])
    .value_counts(dropna=False)
    .rename("count")
    .reset_index()
    .assign(ratio_pct=lambda table: (table["count"] / len(raw_test_dataframe) * 100).round(2))
)

standardized_label_distribution = (
    test_dataframe["label"]
    .value_counts(dropna=False)
    .rename_axis("standardized_label")
    .reset_index(name="count")
    .assign(ratio_pct=lambda table: (table["count"] / len(test_dataframe) * 100).round(2))
)

class_support = standardized_label_distribution.set_index("standardized_label")["count"].to_dict()
unexpected_label_count = int(~test_dataframe["label"].isin([positive_label, negative_label]).sum())
label_readiness = pd.DataFrame(
    [
        {
            "gate": "label_contract",
            "positive_label": positive_label,
            "negative_label": negative_label,
            "unexpected_label_count": unexpected_label_count,
            "positive_support": int(class_support.get(positive_label, 0)),
            "negative_support": int(class_support.get(negative_label, 0)),
            "status": "pass" if unexpected_label_count == 0 and all(class_support.get(label, 0) > 0 for label in [positive_label, negative_label]) else "hold",
            "qa_judgment": "FP/FN 계산에 필요한 label 기준을 사용할 수 있습니다."
            if unexpected_label_count == 0 and all(class_support.get(label, 0) > 0 for label in [positive_label, negative_label])
            else "metric 계산 전에 label 기준과 class support를 확인해야 합니다.",
        }
    ]
)

label_transformation_audit = pd.DataFrame(
    [
        {"step": "copy_before_mutation", "code_pattern": "raw_test_dataframe.copy()", "qa_reason": "원본 label과 표준화 label을 비교할 수 있게 보존합니다."},
        {"step": "standardize_label", "code_pattern": "Series.map(aiq_lite.normalize_label)", "qa_reason": "정답 기준을 high_risk/low_risk 계약으로 맞춥니다."},
        {"step": "check_class_support", "code_pattern": "value_counts(dropna=False)", "qa_reason": "Precision/Recall 해석에 필요한 class 존재 여부를 확인합니다."},
    ]
)


### 출력 확인: `label_transition`

**확인할 점**  
원본 label이 표준화 후 어떤 값으로 바뀌었는지 봅니다.

**왜 중요한가**  
label은 모델 평가의 정답입니다. 같은 뜻의 값이 여러 표기로 섞이면 정답 기준이 흔들립니다.

**다음 단계**  
표준화 결과가 `high_risk`, `low_risk`로 정리됐는지 확인합니다.

**주의할 점**  
표준화 결과만 보지 말고 원본 값도 같이 봅니다.


In [ ]:
display(label_transition)


### 출력 확인: `standardized_label_distribution`

**확인할 점**  
`high_risk`와 `low_risk`의 개수와 비율을 봅니다.

**왜 중요한가**  
`high_risk`는 관심 class입니다. 이 수가 적으면 recall 같은 지표가 크게 흔들릴 수 있습니다.

**다음 단계**  
두 class가 모두 있는지 확인한 뒤 label gate로 넘어갑니다.


In [ ]:
display(standardized_label_distribution)


### 출력 확인: `label_readiness`

**확인할 점**  
invalid label, missing label, class별 표본 수를 봅니다.

**왜 중요한가**  
정답 label에 문제가 있으면 모델 예측을 제대로 채점할 수 없습니다.

**다음 단계**  
label 기준이 유지되면 feature와 metric 해석으로 넘어갑니다.

**주의할 점**  
label gate가 통과해도 feature 품질까지 통과한 것은 아닙니다.


In [ ]:
display(label_readiness)


### 출력 확인: `label_transformation_audit`

**확인할 점**  
원본 label이 표준화 후 어떤 값으로 바뀌었는지 봅니다.

**왜 중요한가**  
label은 모델 평가의 정답입니다. 같은 뜻의 값이 여러 표기로 섞이면 정답 기준이 흔들립니다.

**다음 단계**  
표준화 결과가 `high_risk`, `low_risk`로 정리됐는지 확인합니다.

**주의할 점**  
표준화 결과만 보지 말고 원본 값도 같이 봅니다.


In [ ]:
display(label_transformation_audit)


이 출력에서 확인할 핵심은 label 변환이 metric 계산 전 단계라는 점입니다. label이 흔들리면 FP와 FN이 모두 흔들리므로, 모델 문제처럼 보이는 지표 변화가 실제로는 정답 기준 문제일 수 있습니다.

이 notebook에서는 label 계약이 통과되어야 score와 prediction을 만들 수 있습니다. 통과하더라도 label 반전 후보 같은 더 깊은 정답 기준 신뢰 문제는 별도 제한 사항으로 남겨야 합니다.

### 2-3. feature readiness와 numeric 변환 전제 확인

이 셀에서는 score를 만들 feature가 실제로 존재하고 숫자로 해석 가능한지 확인하는 것입니다. score 산식은 feature 값을 사용하므로, feature 결측이나 숫자 변환 실패는 metric 변화의 원인 후보가 될 수 있습니다.

여기서는 모델 산식으로 바로 넘어가지 않습니다. 먼저 feature 컬럼 존재 여부, 결측 수, 숫자 변환 후 결측 수, 기본 분포를 표로 확인합니다. 이 표는 “metric을 계산해도 되는가”와 “나중에 metric이 흔들릴 때 어떤 feature를 의심할 것인가”를 동시에 정리합니다.

feature readiness는 모델 품질 결론이 아닙니다. 다만 결측이나 범위 이상이 보이면 score 분포와 prediction 분포를 해석할 때 데이터 조건 변화 후보로 남깁니다.


### 따라하기 안내: feature readiness

**이 셀에서 하는 일**  
평가에 필요한 feature가 준비됐는지 확인합니다.

**확인할 점**  
누락 feature, 결측, 숫자 변환 가능 여부를 봅니다.

**왜 중요한가**  
feature가 준비되지 않으면 모델 문제가 아니라 입력 조건 문제일 수 있습니다.


In [ ]:
available_feature_columns = [column for column in feature_columns if column in test_dataframe.columns]
feature_values = test_dataframe.loc[:, available_feature_columns].apply(pd.to_numeric, errors="coerce")

feature_readiness_rows: list[dict[str, object]] = []
for column in feature_columns:
    if column not in test_dataframe.columns:
        feature_readiness_rows.append(
            {
                "feature": column,
                "exists": False,
                "dtype": "missing",
                "missing_count": len(test_dataframe),
                "numeric_na_after_coerce": len(test_dataframe),
                "min": None,
                "median": None,
                "max": None,
                "status": "hold",
            }
        )
        continue
    numeric_values = pd.to_numeric(test_dataframe[column], errors="coerce")
    missing_count = int(test_dataframe[column].isna().sum())
    numeric_na_count = int(numeric_values.isna().sum())
    feature_readiness_rows.append(
        {
            "feature": column,
            "exists": True,
            "dtype": str(test_dataframe[column].dtype),
            "missing_count": missing_count,
            "numeric_na_after_coerce": numeric_na_count,
            "min": round(float(numeric_values.min()), 4) if numeric_values.notna().any() else None,
            "median": round(float(numeric_values.median()), 4) if numeric_values.notna().any() else None,
            "max": round(float(numeric_values.max()), 4) if numeric_values.notna().any() else None,
            "status": "pass" if missing_count == 0 and numeric_na_count == 0 else "review",
        }
    )

feature_readiness = pd.DataFrame(feature_readiness_rows)
feature_contract = pd.DataFrame(
    [
        {
            "gate": "feature_readiness",
            "feature_columns_present": f"{int(feature_readiness['exists'].sum())}/{len(feature_columns)}",
            "features_with_missing": int((feature_readiness["missing_count"] > 0).sum()),
            "features_with_numeric_issue": int((feature_readiness["numeric_na_after_coerce"] > feature_readiness["missing_count"]).sum()),
            "threshold_to_apply_later": operating_threshold,
            "status": "pass" if bool(feature_readiness["exists"].all()) else "hold",
            "qa_judgment": "score 생성에 필요한 feature 컬럼이 준비되었습니다."
            if bool(feature_readiness["exists"].all())
            else "feature 컬럼 누락이 있어 score와 metric 계산을 보류해야 합니다.",
        }
    ]
)

feature_distribution_snapshot = (
    feature_values.describe(percentiles=[0.1, 0.5, 0.9])
    .T
    .loc[:, ["count", "mean", "std", "min", "10%", "50%", "90%", "max"]]
    .round(4)
    .reset_index()
    .rename(columns={"index": "feature"})
)


### 출력 확인: `feature_contract`

**확인할 점**  
필수 컬럼과 feature가 빠지지 않았는지 봅니다.

**왜 중요한가**  
필수 feature가 없으면 score 계산 조건이 달라지고, label이 없으면 metric을 계산할 수 없습니다.

**다음 단계**  
컬럼이 있으면 결측과 값 범위를 따로 확인합니다.

**주의할 점**  
컬럼 존재는 값 품질을 보장하지 않습니다.


In [ ]:
display(feature_contract)


### 출력 확인: `feature_readiness`

**확인할 점**  
feature별 누락, 결측, invalid 상태를 봅니다.

**왜 중요한가**  
문제가 있는 feature는 score와 metric 변화의 원인 후보가 됩니다.

**다음 단계**  
문제가 있는 feature 이름을 기억하고 score/metric 변화와 연결합니다.

**주의할 점**  
바로 모델 결함으로 단정하지 않습니다.


In [ ]:
display(feature_readiness)


### 출력 확인: `feature_distribution_snapshot`

**확인할 점**  
feature별 평균, 최소, 최대 또는 분포 변화를 봅니다.

**왜 중요한가**  
입력 feature 분포가 바뀌면 같은 모델도 다른 score를 낼 수 있습니다.

**다음 단계**  
score 분포 변화와 같은 방향인지 확인합니다.


In [ ]:
display(feature_distribution_snapshot)


이 출력에서 확인할 핵심은 score 계산의 입력 전제가 분리되어 있다는 점입니다. `feature_contract`가 통과되어야 뒤의 score가 “같은 feature 목록에서 계산된 값”이라고 말할 수 있습니다.

분포 snapshot은 정상/비정상 판정표가 아니라 비교 기준입니다. 뒤에서 validation baseline과 degraded 데이터를 비교할 때 같은 feature의 결측, 범위, 중앙값 변화가 score와 metric 변화의 원인 후보가 되는지 다시 연결합니다.

## 3. score 생성과 prediction 생성

### 3-1. feature에서 score 만들기

이 셀에서는 준비된 feature가 어떻게 `score`로 바뀌는지 확인하는 것입니다. score는 아직 최종 예측이 아니며, Positive class 가능성을 나타내는 모델 출력입니다. 이 notebook에서는 Lite에서도 실행 가능한 고정 scoring rule을 사용합니다.

모델 산식 자체를 성능 개선 대상으로 다루지 않습니다. QA가 확인할 것은 같은 feature 목록과 같은 scoring rule로 score가 생성되었는지, 그리고 score 분포가 이후 prediction과 metric 해석에 어떤 전제가 되는지입니다.

이 셀은 숫자 변환, 결측 대체, clip, component 합산을 transformation audit으로 남깁니다. 나중에 score 분포가 달라졌을 때 어떤 처리 단계가 영향을 줄 수 있는지 설명하기 위해서입니다.


### 따라하기 안내: score 계산

**이 셀에서 하는 일**  
feature 값을 이용해 각 행의 `high_risk` score를 계산합니다.

**확인할 점**  
score가 0과 1 사이의 값인지, 어떤 feature가 score 계산에 영향을 주는지 봅니다.

**왜 중요한가**  
score는 아직 최종 예측이 아닙니다. threshold를 적용하기 전의 중간값이라는 점을 구분해야 합니다.


In [ ]:
score_components = pd.DataFrame(
    {
        "heart_rate_component": ((feature_values["heart_rate"].fillna(75) - 60).clip(lower=0, upper=39) / 39 * 0.70),
        "respiratory_component": ((feature_values["respiratory_rate"].fillna(15) - 12).clip(lower=0, upper=7) / 7 * 0.06),
        "temperature_component": ((feature_values["body_temperature"].fillna(36.7) - 36.0).clip(lower=0, upper=2) / 2 * 0.04),
        "oxygen_component": ((98.5 - feature_values["oxygen_saturation"].fillna(98)).clip(lower=0, upper=4) / 4 * 0.06),
        "systolic_component": ((feature_values["systolic_blood_pressure"].fillna(124) - 115).clip(lower=0, upper=25) / 25 * 0.04),
    }
)

scored_dataframe = test_dataframe.copy().assign(
    score=(0.10 + score_components.sum(axis=1)).clip(lower=0.01, upper=0.99).round(4)
)

score_summary = (
    scored_dataframe["score"]
    .describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])
    .round(4)
    .rename("score")
    .reset_index()
    .rename(columns={"index": "stat"})
)

component_summary = (
    score_components.agg(["mean", "max"])
    .T
    .round(4)
    .reset_index()
    .rename(columns={"index": "component"})
)
score_transformation_audit = pd.DataFrame(
    [
        {"step": "numeric_feature_values", "operation": "DataFrame.apply(pd.to_numeric, errors='coerce')", "qa_reason": "feature가 score 산식에 들어가기 전에 숫자 값인지 확인합니다."},
        {"step": "missing_default", "operation": "Series.fillna(...) per feature", "qa_reason": "결측이 score에 들어갈 때 어떤 기본값이 쓰였는지 남깁니다."},
        {"step": "bounded_component", "operation": "Series.clip(lower=..., upper=...)", "qa_reason": "극단값이 component를 과도하게 흔들지 않도록 제한한 조건을 기록합니다."},
        {"step": "score", "operation": "0.10 + component sum, clipped to [0.01, 0.99]", "qa_reason": "prediction 이전의 연속값을 만듭니다."},
    ]
)

score_preview_columns = ["patient_id", *feature_columns, "label", "score"]


### 출력 확인: `score_transformation_audit`

**확인할 점**  
데이터셋, threshold, positive label, model 조건을 봅니다.

**왜 중요한가**  
metric은 조건 없이 해석할 수 없습니다. threshold가 달라지면 FP/FN도 달라집니다.

**다음 단계**  
조건을 확인한 뒤 metric이나 prediction 결과를 읽습니다.


In [ ]:
display(score_transformation_audit)


### 출력 확인: `scored_dataframe.loc[:, score_preview_columns].head()`

**확인할 점**  
몇 개 행을 직접 보고 컬럼 이름과 값 모양이 예상과 맞는지 봅니다.

**왜 중요한가**  
Preview는 전체 확인이 아니라 데이터 모양 확인입니다.

**다음 단계**  
분포나 metric은 뒤의 요약 표에서 확인합니다.


In [ ]:
display(scored_dataframe.loc[:, score_preview_columns].head())


### 출력 확인: `score_summary`

**확인할 점**  
score의 평균, 범위, 분위수 또는 구성 요소를 봅니다.

**왜 중요한가**  
score는 prediction 전 단계입니다. score가 움직이면 입력 분포나 모델 조건 변화를 의심할 수 있습니다.

**다음 단계**  
prediction 분포와 metric 변화가 같은 방향인지 확인합니다.

**주의할 점**  
score를 실제 확률이라고 단정하지 않습니다.


In [ ]:
display(score_summary)


### 출력 확인: `component_summary`

**확인할 점**  
score의 평균, 범위, 분위수 또는 구성 요소를 봅니다.

**왜 중요한가**  
score는 prediction 전 단계입니다. score가 움직이면 입력 분포나 모델 조건 변화를 의심할 수 있습니다.

**다음 단계**  
prediction 분포와 metric 변화가 같은 방향인지 확인합니다.

**주의할 점**  
score를 실제 확률이라고 단정하지 않습니다.


In [ ]:
display(component_summary)


이 출력에서 확인할 핵심은 `score`가 `label`이나 `prediction`이 아니라는 점입니다. score는 feature에서 생성된 연속값이며, 아직 `high_risk` 또는 `low_risk`라는 최종 class가 아닙니다.

component summary는 어떤 입력 feature가 score에 영향을 주는지 보여줍니다. 점수 분포가 기준과 달라지면 입력 feature 분포 변화 또는 결측/범위 오류를 원인 후보로 볼 수 있습니다.

### 3-2. threshold를 적용해 prediction 만들기

이 셀에서는 같은 score라도 threshold에 따라 prediction이 달라진다는 점을 확인하는 것입니다. threshold는 운영 기준이며, 모델이 새로 학습되는 것이 아닙니다.

기본 threshold는 `0.50`입니다. score가 threshold 이상이면 `high_risk`, 그렇지 않으면 `low_risk`로 prediction을 만듭니다. 이 변환을 분리해서 보여야 뒤에서 threshold sweep을 해석할 수 있습니다.


### 따라하기 안내: threshold 적용

**이 셀에서 하는 일**  
score를 prediction으로 바꿉니다.

**확인할 점**  
threshold 값과 prediction 분포를 확인합니다.

**왜 중요한가**  
threshold는 운영 기준입니다. 같은 score라도 threshold가 달라지면 FP/FN이 달라집니다.


In [ ]:
prediction_dataframe = scored_dataframe.assign(
    prediction=lambda table: table["score"].ge(operating_threshold).map(
        {True: positive_label, False: negative_label}
    ),
    is_correct=lambda table: table["label"].eq(table["prediction"]),
)

prediction_distribution = (
    prediction_dataframe["prediction"]
    .value_counts(dropna=False)
    .rename_axis("prediction")
    .reset_index(name="count")
    .assign(ratio_pct=lambda table: (table["count"] / len(prediction_dataframe) * 100).round(2))
)

prediction_transformation_audit = pd.DataFrame(
    [
        {"step": "threshold_fixed", "observed": operating_threshold, "qa_reason": "같은 score를 같은 운영 기준으로 class로 변환합니다."},
        {"step": "prediction_rule", "observed": "score >= threshold -> high_risk else low_risk", "qa_reason": "prediction 분포와 label 분포를 분리해서 해석합니다."},
    ]
)

label_prediction_preview = prediction_dataframe.loc[
    :, ["patient_id", "label", "score", "prediction", "is_correct"]
].head(10)


### 출력 확인: `prediction_transformation_audit`

**확인할 점**  
데이터셋, threshold, positive label, model 조건을 봅니다.

**왜 중요한가**  
metric은 조건 없이 해석할 수 없습니다. threshold가 달라지면 FP/FN도 달라집니다.

**다음 단계**  
조건을 확인한 뒤 metric이나 prediction 결과를 읽습니다.


In [ ]:
display(prediction_transformation_audit)


### 출력 확인: `label_prediction_preview`

**확인할 점**  
몇 개 행을 직접 보고 컬럼 이름과 값 모양이 예상과 맞는지 봅니다.

**왜 중요한가**  
Preview는 전체 확인이 아니라 데이터 모양 확인입니다.

**다음 단계**  
분포나 metric은 뒤의 요약 표에서 확인합니다.


In [ ]:
display(label_prediction_preview)


### 출력 확인: `prediction_distribution`

**확인할 점**  
`high_risk`, `low_risk` 예측 비율을 봅니다.

**왜 중요한가**  
prediction은 score에 threshold를 적용한 결과입니다.

**다음 단계**  
score 변화와 threshold 기준을 함께 보고 해석합니다.

**주의할 점**  
prediction 분포는 정답과 비교한 성능 지표가 아닙니다.


In [ ]:
display(prediction_distribution)


이 출력에서 확인할 핵심은 label 분포와 prediction 분포가 서로 다른 값이라는 점입니다. label은 정답 기준이고 prediction은 score와 threshold로 만든 모델 출력입니다.

`high_risk` prediction 비율이 높아졌다는 관측은 이 단계 이후에야 말할 수 있습니다. 1장의 label 분포와 2장의 prediction 분포를 섞으면 원인 후보를 잘못 좁히게 됩니다.

## 4. Confusion Matrix와 metric 계산

### 4-1. label과 prediction을 비교해 오류 유형 만들기

이 셀에서는 prediction이 맞았는지 틀렸는지를 TP, TN, FP, FN으로 나누는 것입니다. Accuracy 하나는 전체 요약이지만, QA 해석에는 FP와 FN의 건수가 반드시 필요합니다.

관심 class는 `high_risk`입니다. FP는 실제 `low_risk`인데 `high_risk`로 예측한 경우이고, FN은 실제 `high_risk`인데 `low_risk`로 예측한 경우입니다.


### 따라하기 안내: confusion matrix

**이 셀에서 하는 일**  
label과 prediction을 비교해 오류 유형을 봅니다.

**확인할 점**  
TP, FP, FN, TN 중 FP와 FN을 특히 봅니다.

**왜 중요한가**  
precision과 recall은 FP/FN과 연결됩니다. accuracy만 보면 오류 유형을 놓칩니다.


In [ ]:
confusion_table = pd.crosstab(
    prediction_dataframe["label"],
    prediction_dataframe["prediction"],
    rownames=["actual_label"],
    colnames=["prediction"],
    dropna=False,
).reindex(index=[positive_label, negative_label], columns=[positive_label, negative_label], fill_value=0)

tp = int(confusion_table.loc[positive_label, positive_label])
fn = int(confusion_table.loc[positive_label, negative_label])
fp = int(confusion_table.loc[negative_label, positive_label])
tn = int(confusion_table.loc[negative_label, negative_label])

confusion_summary = pd.DataFrame(
    [
        {"type": "TP", "count": tp, "meaning": "actual high_risk, predicted high_risk"},
        {"type": "FP", "count": fp, "meaning": "actual low_risk, predicted high_risk"},
        {"type": "FN", "count": fn, "meaning": "actual high_risk, predicted low_risk"},
        {"type": "TN", "count": tn, "meaning": "actual low_risk, predicted low_risk"},
    ]
)

accuracy = (tp + tn) / len(prediction_dataframe) if len(prediction_dataframe) else 0.0
precision = tp / (tp + fp) if (tp + fp) else 0.0
recall = tp / (tp + fn) if (tp + fn) else 0.0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

metric_table = pd.DataFrame(
    [
        {
            "dataset": "vital_signs_test",
            "threshold": operating_threshold,
            "row_count": len(prediction_dataframe),
            "accuracy": round(accuracy, 4),
            "precision": round(precision, 4),
            "recall": round(recall, 4),
            "f1": round(f1, 4),
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "TN": tn,
        }
    ]
)

metric_input_audit = pd.DataFrame(
    [
        {"input": "label", "source": "standardized test_dataframe.label", "qa_reason": "정답 기준입니다."},
        {"input": "score", "source": "scored_dataframe.score", "qa_reason": "threshold 적용 전 모델 출력입니다."},
        {"input": "prediction", "source": "prediction_dataframe.prediction", "qa_reason": "score와 threshold로 만든 최종 예측입니다."},
        {"input": "threshold", "source": operating_threshold, "qa_reason": "metric 계산 조건으로 기록해야 합니다."},
    ]
)


### 출력 확인: `metric_input_audit`

**확인할 점**  
데이터셋, threshold, positive label, model 조건을 봅니다.

**왜 중요한가**  
metric은 조건 없이 해석할 수 없습니다. threshold가 달라지면 FP/FN도 달라집니다.

**다음 단계**  
조건을 확인한 뒤 metric이나 prediction 결과를 읽습니다.


In [ ]:
display(metric_input_audit)


### 출력 확인: `confusion_table`

**확인할 점**  
TP, FP, FN, TN을 봅니다. 특히 FP와 FN을 구분합니다.

**왜 중요한가**  
FP는 low risk를 high risk로 잘못 알린 경우이고, FN은 high risk를 놓친 경우입니다. precision과 recall은 이 오류와 연결됩니다.

**다음 단계**  
FP/FN 중 어떤 오류가 더 큰지 metric table과 함께 봅니다.

**주의할 점**  
actual과 prediction 축을 바꾸어 읽으면 해석이 반대로 됩니다.


In [ ]:
display(confusion_table)


### 출력 확인: `confusion_summary`

**확인할 점**  
TP, FP, FN, TN을 봅니다. 특히 FP와 FN을 구분합니다.

**왜 중요한가**  
FP는 low risk를 high risk로 잘못 알린 경우이고, FN은 high risk를 놓친 경우입니다. precision과 recall은 이 오류와 연결됩니다.

**다음 단계**  
FP/FN 중 어떤 오류가 더 큰지 metric table과 함께 봅니다.

**주의할 점**  
actual과 prediction 축을 바꾸어 읽으면 해석이 반대로 됩니다.


In [ ]:
display(confusion_summary)


### 출력 확인: `metric_table`

**확인할 점**  
accuracy, precision, recall, F1, AUROC/PR-AUC를 봅니다.

**왜 중요한가**  
precision은 FP의 영향을 받고, recall은 FN의 영향을 받습니다. accuracy 하나로는 오류 유형을 알 수 없습니다.

**다음 단계**  
confusion matrix의 FP/FN과 함께 해석합니다.

**주의할 점**  
metric에는 데이터셋과 threshold 조건을 같이 붙여야 합니다.


In [ ]:
display(metric_table)


이 출력에서 확인할 핵심은 오류 유형입니다. Precision은 FP의 영향을 받고 Recall은 FN의 영향을 받습니다. 따라서 Accuracy가 높거나 낮다는 문장만으로는 운영 리스크를 설명할 수 없습니다.

보고서에는 metric 값과 함께 threshold, row count, FP, FN을 같이 남깁니다. 그래야 reviewer가 같은 데이터와 같은 기준에서 계산한 값인지 추적할 수 있습니다.

## 5. 기준 validation과 품질 저하 validation 비교

### 5-1. validation 원본 데이터 두 개를 metric 전에 먼저 비교

이 셀에서는 metric delta를 보기 전에 baseline과 degraded 데이터가 어떤 입력 조건을 갖는지 확인하는 것입니다. 같은 모델과 같은 threshold를 적용하더라도 데이터 조건이 달라지면 score, prediction, metric이 달라질 수 있습니다.

`valid_baseline`은 기준 validation 조건이고, `valid_degraded`는 현재 운영 입력에서 나타날 수 있는 결측, 범위 오류, 라벨 기준 흔들림을 교육용으로 재현한 비교 데이터입니다. 이 비교는 실제 운영 로그 자체가 아니라, 같은 평가 조건에서 데이터 품질 후보를 분리해 보기 위한 validation evidence입니다.

이 단계에서는 아직 score를 만들지 않습니다. 먼저 source, row count, raw label 분포, preview를 나란히 확인해 “무엇이 비교되고 있는가”를 고정합니다.


### 따라하기 안내: validation 비교 준비

**이 셀에서 하는 일**  
validation 비교 조건을 준비합니다.

**확인할 점**  
비교 데이터셋과 고정할 model/threshold를 확인합니다.

**왜 중요한가**  
모델과 threshold를 고정해야 데이터 조건 변화의 영향을 볼 수 있습니다.


In [ ]:
VALIDATION_DATASETS = [
    ("valid_baseline", "data/vital_signs_valid_baseline.csv"),
    ("valid_degraded", "data/vital_signs_valid_degraded.csv"),
]

raw_validation_frames: dict[str, pd.DataFrame] = {}
validation_source_rows: list[dict[str, object]] = []
validation_preview_frames: list[pd.DataFrame] = []
validation_raw_label_frames: list[pd.DataFrame] = []

for dataset_name, dataset_path in VALIDATION_DATASETS:
    raw_frame, source = aiq_lite.load_csv_or_sample(dataset_path, aiq_lite.sample_vital_signs(), nrows=None)
    raw_validation_frames[dataset_name] = raw_frame
    scope = "embedded_fallback_sample" if "embedded" in source else (
        "jupyterlite_sample" if len(raw_frame) <= 600 else "local_full_or_large_sample"
    )
    validation_source_rows.append(
        {
            "dataset": dataset_name,
            "path": dataset_path,
            "data_source": source,
            "execution_scope": scope,
            "row_grain": "one validation classification sample",
            "row_count": raw_frame.shape[0],
            "column_count": raw_frame.shape[1],
        }
    )
    validation_preview_frames.append(
        raw_frame.loc[:, raw_preview_columns]
        .head(3)
        .assign(dataset=dataset_name)
        .loc[:, ["dataset", *raw_preview_columns]]
    )
    validation_raw_label_frames.append(
        raw_frame["label"]
        .value_counts(dropna=False)
        .rename_axis("raw_label")
        .reset_index(name="count")
        .assign(
            dataset=dataset_name,
            ratio_pct=lambda table, denominator=len(raw_frame): (table["count"] / denominator * 100).round(2),
        )
    )

validation_source_table = pd.DataFrame(validation_source_rows)
validation_raw_preview = pd.concat(validation_preview_frames, ignore_index=True)
validation_raw_label_distribution = pd.concat(validation_raw_label_frames, ignore_index=True).loc[
    :, ["dataset", "raw_label", "count", "ratio_pct"]
]

validation_comparison_brief = pd.DataFrame(
    [
        {"check": "same_row_grain", "observed": "one validation classification sample", "qa_use": "두 데이터셋의 metric 분모가 같은 의미인지 확인합니다."},
        {"check": "score_status", "observed": "not_created_yet", "qa_use": "metric 전에 데이터 조건 차이를 먼저 확인합니다."},
        {"check": "comparison_limit", "observed": "validation evidence, not live operation log", "qa_use": "운영 원인 확정이 아니라 후보 강화 근거로 사용합니다."},
    ]
)


### 출력 확인: `validation_source_table`

**확인할 점**  
데이터나 artifact가 어디에서 왔는지 봅니다. 경로와 실행 범위를 먼저 확인합니다.

**왜 중요한가**  
같은 숫자라도 prepared artifact인지 local 재생성인지에 따라 보고서 표현이 달라집니다.

**다음 단계**  
이 범위를 기억한 상태로 다음 표를 읽습니다.


In [ ]:
display(validation_source_table)


### 출력 확인: `validation_comparison_brief`

**확인할 점**  
표의 항목과 값을 천천히 봅니다.

**왜 중요한가**  
이 출력은 바로 앞 코드 셀이 만든 단일 근거입니다.

**다음 단계**  
다음 셀에서 이 값을 어떻게 쓰는지 확인합니다.


In [ ]:
display(validation_comparison_brief)


### 출력 확인: `validation_raw_preview`

**확인할 점**  
몇 개 행을 직접 보고 컬럼 이름과 값 모양이 예상과 맞는지 봅니다.

**왜 중요한가**  
Preview는 전체 확인이 아니라 데이터 모양 확인입니다.

**다음 단계**  
분포나 metric은 뒤의 요약 표에서 확인합니다.


In [ ]:
display(validation_raw_preview)


### 출력 확인: `validation_raw_label_distribution`

**확인할 점**  
`high_risk`와 `low_risk`의 개수와 비율을 봅니다.

**왜 중요한가**  
`high_risk`는 관심 class입니다. 이 수가 적으면 recall 같은 지표가 크게 흔들릴 수 있습니다.

**다음 단계**  
두 class가 모두 있는지 확인한 뒤 label gate로 넘어갑니다.


In [ ]:
display(validation_raw_label_distribution)


이 출력에서 확인할 핵심은 baseline과 degraded가 같은 의미의 validation sample을 비교한다는 점입니다. row grain이 다르면 metric delta는 데이터 조건 변화가 아니라 비교 대상 차이를 반영할 수 있습니다.

raw label 분포는 score 이전의 정답 기준 상태입니다. 허용 label 값만 보인다고 해서 정답 기준이 완전히 신뢰된다는 뜻은 아니지만, 최소한 metric 계산 형식은 유지되는지 확인할 수 있습니다.

### 5-2. validation label 표준화와 feature 품질 신호 비교

이 셀에서는 degraded 데이터가 metric 계산 전에 어떤 품질 신호를 갖는지 확인하는 것입니다. metric delta를 먼저 보면 “모델이 나빠졌다”로 단정하기 쉽지만, 결측, 범위 오류, label 분포 차이가 먼저 보이면 데이터 조건 변화 후보를 남길 수 있습니다.

두 데이터셋 모두 같은 label 표준화와 같은 feature readiness 기준을 적용합니다. baseline에는 없는 결측이나 범위 이상이 degraded에 나타나면, 뒤에서 score 분포와 FP/FN 변화의 원인 후보로 연결합니다.

이 비교가 증명하는 것은 “데이터 조건 차이가 관측되었다”는 점입니다. 이 비교만으로 모델 자체 결함이나 실제 운영 장애를 확정하지 않고, score/prediction/metric 변화와 함께 해석합니다.


### 따라하기 안내: validation 데이터 로딩

**이 셀에서 하는 일**  
validation 데이터를 읽고 기본 상태를 봅니다.

**확인할 점**  
행 수, label 분포, preview를 확인합니다.

**왜 중요한가**  
데이터를 같은 방식으로 읽어야 공정하게 비교할 수 있습니다.


In [ ]:
validation_frames: dict[str, pd.DataFrame] = {}
validation_label_transition_frames: list[pd.DataFrame] = []
validation_label_distribution_frames: list[pd.DataFrame] = []
feature_quality_rows: list[dict[str, object]] = []
range_quality_rows: list[dict[str, object]] = []

for dataset_name, raw_frame in raw_validation_frames.items():
    standardized_frame = raw_frame.copy().assign(
        label=lambda table: table["label"].map(aiq_lite.normalize_label),
        dataset=dataset_name,
    )
    validation_frames[dataset_name] = standardized_frame

    validation_label_transition_frames.append(
        raw_frame.loc[:, ["label"]]
        .rename(columns={"label": "raw_label"})
        .assign(standardized_label=standardized_frame["label"], dataset=dataset_name)
        .value_counts(dropna=False)
        .rename("count")
        .reset_index()
    )
    validation_label_distribution_frames.append(
        standardized_frame["label"]
        .value_counts(dropna=False)
        .rename_axis("standardized_label")
        .reset_index(name="count")
        .assign(
            dataset=dataset_name,
            ratio_pct=lambda table, denominator=len(standardized_frame): (table["count"] / denominator * 100).round(2),
        )
    )

    numeric_features = standardized_frame.loc[:, feature_columns].apply(pd.to_numeric, errors="coerce")
    rows_with_any_feature_missing = int(standardized_frame.loc[:, feature_columns].isna().any(axis=1).sum())
    feature_quality_rows.append(
        {
            "dataset": dataset_name,
            "row_count": len(standardized_frame),
            "rows_with_any_feature_missing": rows_with_any_feature_missing,
            "missing_row_pct": round(rows_with_any_feature_missing / len(standardized_frame) * 100, 2),
            "heart_rate_median": round(float(numeric_features["heart_rate"].median()), 4),
            "oxygen_saturation_min": round(float(numeric_features["oxygen_saturation"].min()), 4),
            "oxygen_saturation_missing": int(standardized_frame["oxygen_saturation"].isna().sum()),
        }
    )

    for column, (minimum, maximum) in aiq_lite.VALID_RANGES.items():
        if column not in standardized_frame.columns:
            continue
        values = pd.to_numeric(standardized_frame[column], errors="coerce")
        invalid_count = int((values.notna() & ((values < minimum) | (values > maximum))).sum())
        range_quality_rows.append(
            {
                "dataset": dataset_name,
                "feature": column,
                "valid_min": minimum,
                "valid_max": maximum,
                "invalid_count": invalid_count,
                "invalid_pct": round(invalid_count / len(standardized_frame) * 100, 2),
            }
        )

validation_label_transition = pd.concat(validation_label_transition_frames, ignore_index=True).loc[
    :, ["dataset", "raw_label", "standardized_label", "count"]
]
validation_label_distribution = pd.concat(validation_label_distribution_frames, ignore_index=True).loc[
    :, ["dataset", "standardized_label", "count", "ratio_pct"]
]
validation_feature_quality = pd.DataFrame(feature_quality_rows)
validation_range_quality = pd.DataFrame(range_quality_rows).sort_values(
    ["invalid_count", "dataset", "feature"], ascending=[False, True, True]
)

quality_delta_columns = ["rows_with_any_feature_missing", "missing_row_pct", "heart_rate_median", "oxygen_saturation_min", "oxygen_saturation_missing"]
quality_by_dataset = validation_feature_quality.set_index("dataset")
validation_quality_delta = (
    (quality_by_dataset.loc["valid_degraded", quality_delta_columns] - quality_by_dataset.loc["valid_baseline", quality_delta_columns])
    .to_frame()
    .T
    .round(4)
    .assign(comparison="valid_degraded_minus_baseline")
    .loc[:, ["comparison", *quality_delta_columns]]
)

validation_data_risk = pd.DataFrame(
    [
        {
            "risk_candidate": "feature_missing_or_range_shift",
            "observed": "degraded feature quality is compared before scoring",
            "can_explain": "score distribution, prediction distribution, FP/FN movement",
            "cannot_prove": "model defect by itself",
            "next_evidence": "score and metric comparison under same threshold",
        }
    ]
)


### 출력 확인: `validation_label_transition`

**확인할 점**  
원본 label이 표준화 후 어떤 값으로 바뀌었는지 봅니다.

**왜 중요한가**  
label은 모델 평가의 정답입니다. 같은 뜻의 값이 여러 표기로 섞이면 정답 기준이 흔들립니다.

**다음 단계**  
표준화 결과가 `high_risk`, `low_risk`로 정리됐는지 확인합니다.

**주의할 점**  
표준화 결과만 보지 말고 원본 값도 같이 봅니다.


In [ ]:
display(validation_label_transition)


### 출력 확인: `validation_label_distribution`

**확인할 점**  
`high_risk`와 `low_risk`의 개수와 비율을 봅니다.

**왜 중요한가**  
`high_risk`는 관심 class입니다. 이 수가 적으면 recall 같은 지표가 크게 흔들릴 수 있습니다.

**다음 단계**  
두 class가 모두 있는지 확인한 뒤 label gate로 넘어갑니다.


In [ ]:
display(validation_label_distribution)


### 출력 확인: `validation_feature_quality`

**확인할 점**  
feature별 누락, 결측, invalid 상태를 봅니다.

**왜 중요한가**  
문제가 있는 feature는 score와 metric 변화의 원인 후보가 됩니다.

**다음 단계**  
문제가 있는 feature 이름을 기억하고 score/metric 변화와 연결합니다.

**주의할 점**  
바로 모델 결함으로 단정하지 않습니다.


In [ ]:
display(validation_feature_quality)


### 출력 확인: `validation_quality_delta`

**확인할 점**  
baseline 대비 증가/감소 방향을 봅니다.

**왜 중요한가**  
delta는 원인 확정이 아니라 어디가 달라졌는지 보여 줍니다.

**다음 단계**  
데이터 품질 delta, score delta, metric delta를 함께 연결합니다.

**주의할 점**  
모든 delta의 부호가 같은 의미는 아닙니다.


In [ ]:
display(validation_quality_delta)


### 출력 확인: `validation_range_quality.head(12)`

**확인할 점**  
허용 범위를 벗어난 값이 있는지 봅니다.

**왜 중요한가**  
범위 오류는 데이터 수집, 단위, 전처리 문제일 수 있고 score를 흔들 수 있습니다.

**다음 단계**  
범위 오류가 있는 feature를 score/metric 변화 후보로 연결합니다.

**주의할 점**  
이상값을 바로 제거한다고 결론 내리지 않습니다.


In [ ]:
display(validation_range_quality.head(12))


### 출력 확인: `validation_data_risk`

**확인할 점**  
표의 항목과 값을 천천히 봅니다.

**왜 중요한가**  
이 출력은 바로 앞 코드 셀이 만든 단일 근거입니다.

**다음 단계**  
다음 셀에서 이 값을 어떻게 쓰는지 확인합니다.


In [ ]:
display(validation_data_risk)


이 출력에서 확인할 핵심은 metric delta보다 앞선 데이터 차이입니다. degraded 데이터에서 결측 행, 범위 이상, 특정 feature 분포 변화가 보이면 score와 prediction 변화의 원인 후보가 강화됩니다.

다만 이 표는 원인을 확정하지 않습니다. 데이터 조건 차이가 있다는 근거를 만든 뒤, 같은 score 산식과 같은 threshold에서 실제 metric 변화가 같은 방향으로 나타나는지 확인해야 합니다.

### 5-3. 같은 scoring rule과 threshold로 validation metric 비교

이 셀에서는 모델 자체를 바꾸지 않았을 때 데이터 조건 변화가 score, prediction, metric에 어떤 차이를 만드는지 확인하는 것입니다. 앞 셀에서 데이터 차이를 먼저 확인했으므로, 이제 metric delta를 데이터 조건 후보와 연결할 수 있습니다.

두 데이터셋에는 같은 feature 목록, 같은 score 산식, 같은 threshold를 적용합니다. 이 조건을 고정해야 metric 변화가 모델 재학습이나 threshold 변경 때문인지 혼동하지 않습니다.

출력은 score 분포, prediction 분포, metric delta를 순서대로 보여줍니다. score와 prediction이 먼저 움직이고 FP/FN이 함께 변하면 데이터 조건 변화 후보를 다음 분석으로 넘길 수 있습니다.


### 따라하기 안내: 같은 기준 적용

**이 셀에서 하는 일**  
baseline validation과 degraded validation에 같은 score 계산 방식과 같은 threshold를 적용합니다.

**확인할 점**  
두 데이터셋이 같은 기준으로 평가되었는지, score 분포와 prediction 분포가 어떻게 달라지는지 봅니다.

**왜 중요한가**  
기준이 같아야 지표 변화가 모델 변경 때문인지 데이터 조건 변화 때문인지 분리해서 볼 수 있습니다.


In [ ]:
def add_scores_and_predictions(frame: pd.DataFrame, threshold: float) -> pd.DataFrame:
    values = frame.loc[:, feature_columns].apply(pd.to_numeric, errors="coerce")
    components = pd.DataFrame(
        {
            "heart_rate_component": ((values["heart_rate"].fillna(75) - 60).clip(lower=0, upper=39) / 39 * 0.70),
            "respiratory_component": ((values["respiratory_rate"].fillna(15) - 12).clip(lower=0, upper=7) / 7 * 0.06),
            "temperature_component": ((values["body_temperature"].fillna(36.7) - 36.0).clip(lower=0, upper=2) / 2 * 0.04),
            "oxygen_component": ((98.5 - values["oxygen_saturation"].fillna(98)).clip(lower=0, upper=4) / 4 * 0.06),
            "systolic_component": ((values["systolic_blood_pressure"].fillna(124) - 115).clip(lower=0, upper=25) / 25 * 0.04),
        }
    )
    return frame.copy().assign(
        score=(0.10 + components.sum(axis=1)).clip(lower=0.01, upper=0.99).round(4),
        prediction=lambda table: table["score"].ge(threshold).map({True: positive_label, False: negative_label}),
    )


def metric_summary(frame: pd.DataFrame, dataset_name: str, threshold: float) -> dict[str, object]:
    table = pd.crosstab(frame["label"], frame["prediction"]).reindex(
        index=[positive_label, negative_label],
        columns=[positive_label, negative_label],
        fill_value=0,
    )
    tp_value = int(table.loc[positive_label, positive_label])
    fn_value = int(table.loc[positive_label, negative_label])
    fp_value = int(table.loc[negative_label, positive_label])
    tn_value = int(table.loc[negative_label, negative_label])
    precision_value = tp_value / (tp_value + fp_value) if (tp_value + fp_value) else 0.0
    recall_value = tp_value / (tp_value + fn_value) if (tp_value + fn_value) else 0.0
    accuracy_value = (tp_value + tn_value) / len(frame) if len(frame) else 0.0
    f1_value = 2 * precision_value * recall_value / (precision_value + recall_value) if (precision_value + recall_value) else 0.0
    return {
        "dataset": dataset_name,
        "threshold": threshold,
        "row_count": len(frame),
        "accuracy": round(accuracy_value, 4),
        "precision": round(precision_value, 4),
        "recall": round(recall_value, 4),
        "f1": round(f1_value, 4),
        "TP": tp_value,
        "FP": fp_value,
        "FN": fn_value,
        "TN": tn_value,
    }

valid_baseline_scored = add_scores_and_predictions(validation_frames["valid_baseline"], operating_threshold)
valid_degraded_scored = add_scores_and_predictions(validation_frames["valid_degraded"], operating_threshold)

validation_scored_frames = {
    "valid_baseline": valid_baseline_scored,
    "valid_degraded": valid_degraded_scored,
}

score_distribution_by_dataset = pd.concat(
    [
        frame["score"]
        .describe(percentiles=[0.1, 0.5, 0.9])
        .rename(dataset_name)
        .to_frame()
        .T
        .assign(dataset=dataset_name)
        for dataset_name, frame in validation_scored_frames.items()
    ],
    ignore_index=True,
).loc[:, ["dataset", "count", "mean", "std", "min", "10%", "50%", "90%", "max"]].round(4)

prediction_distribution_by_dataset = pd.concat(
    [
        frame["prediction"]
        .value_counts(dropna=False)
        .rename_axis("prediction")
        .reset_index(name="count")
        .assign(
            dataset=dataset_name,
            ratio_pct=lambda table, denominator=len(frame): (table["count"] / denominator * 100).round(2),
        )
        for dataset_name, frame in validation_scored_frames.items()
    ],
    ignore_index=True,
).loc[:, ["dataset", "prediction", "count", "ratio_pct"]]

validation_metrics = pd.DataFrame(
    [
        metric_summary(valid_baseline_scored, "valid_baseline", operating_threshold),
        metric_summary(valid_degraded_scored, "valid_degraded", operating_threshold),
    ]
)
metric_columns = ["accuracy", "precision", "recall", "f1", "FP", "FN"]
metric_by_dataset = validation_metrics.set_index("dataset")
metric_delta = (
    (metric_by_dataset.loc["valid_degraded", metric_columns] - metric_by_dataset.loc["valid_baseline", metric_columns])
    .to_frame()
    .T
    .round(4)
    .assign(comparison="valid_degraded_minus_baseline")
    .loc[:, ["comparison", *metric_columns]]
)

validation_metric_audit = pd.DataFrame(
    [
        {"condition": "same_feature_columns", "observed": f"{len(feature_columns)} features", "qa_reason": "입력 feature 조건을 고정합니다."},
        {"condition": "same_scoring_rule", "observed": "browser_safe_score_components_v1", "qa_reason": "모델 조건 변화와 데이터 조건 변화를 섞지 않습니다."},
        {"condition": "same_threshold", "observed": operating_threshold, "qa_reason": "threshold 변경 효과를 제외합니다."},
    ]
)


### 출력 확인: `validation_metric_audit`

**확인할 점**  
데이터셋, threshold, positive label, model 조건을 봅니다.

**왜 중요한가**  
metric은 조건 없이 해석할 수 없습니다. threshold가 달라지면 FP/FN도 달라집니다.

**다음 단계**  
조건을 확인한 뒤 metric이나 prediction 결과를 읽습니다.


In [ ]:
display(validation_metric_audit)


### 출력 확인: `score_distribution_by_dataset`

**확인할 점**  
score의 평균, 범위, 분위수 또는 구성 요소를 봅니다.

**왜 중요한가**  
score는 prediction 전 단계입니다. score가 움직이면 입력 분포나 모델 조건 변화를 의심할 수 있습니다.

**다음 단계**  
prediction 분포와 metric 변화가 같은 방향인지 확인합니다.

**주의할 점**  
score를 실제 확률이라고 단정하지 않습니다.


In [ ]:
display(score_distribution_by_dataset)


### 출력 확인: `prediction_distribution_by_dataset`

**확인할 점**  
`high_risk`, `low_risk` 예측 비율을 봅니다.

**왜 중요한가**  
prediction은 score에 threshold를 적용한 결과입니다.

**다음 단계**  
score 변화와 threshold 기준을 함께 보고 해석합니다.

**주의할 점**  
prediction 분포는 정답과 비교한 성능 지표가 아닙니다.


In [ ]:
display(prediction_distribution_by_dataset)


### 출력 확인: `validation_metrics`

**확인할 점**  
accuracy, precision, recall, F1, AUROC/PR-AUC를 봅니다.

**왜 중요한가**  
precision은 FP의 영향을 받고, recall은 FN의 영향을 받습니다. accuracy 하나로는 오류 유형을 알 수 없습니다.

**다음 단계**  
confusion matrix의 FP/FN과 함께 해석합니다.

**주의할 점**  
metric에는 데이터셋과 threshold 조건을 같이 붙여야 합니다.


In [ ]:
display(validation_metrics)


### 출력 확인: `metric_delta`

**확인할 점**  
accuracy, precision, recall, F1, AUROC/PR-AUC를 봅니다.

**왜 중요한가**  
precision은 FP의 영향을 받고, recall은 FN의 영향을 받습니다. accuracy 하나로는 오류 유형을 알 수 없습니다.

**다음 단계**  
confusion matrix의 FP/FN과 함께 해석합니다.

**주의할 점**  
metric에는 데이터셋과 threshold 조건을 같이 붙여야 합니다.


In [ ]:
display(metric_delta)


이 출력에서 확인할 핵심은 같은 scoring rule과 같은 threshold에서 어떤 metric이 변했는가입니다. Precision이 낮아지거나 FP가 증가하면 오탐 증가 후보를 남기고, Recall이 낮아지거나 FN이 증가하면 미탐 증가 후보를 남깁니다.

이 비교만으로 모델 자체 결함을 확정하지 않습니다. 앞에서 확인한 데이터 조건 차이가 score와 prediction 분포를 거쳐 metric 변화와 함께 나타나는지 확인한 것이므로, 다음 notebook에서는 데이터 품질 신호와 metric 변화를 더 직접 연결해야 합니다.

## 6. threshold sweep과 오류 trade-off

### 6-1. 같은 score에 threshold만 바꿔 적용

이 셀에서는 threshold 변경이 모델 재학습이 아니라 운영 기준 변경이라는 점을 확인하는 것입니다. 같은 score라도 threshold를 낮추면 `high_risk` prediction이 늘고, threshold를 높이면 줄어듭니다.

threshold sweep은 지표를 좋게 보이게 만드는 작업이 아닙니다. FP와 FN 중 어떤 오류가 더 부담인지 확인하기 위한 trade-off evidence입니다.


### 따라하기 안내: threshold 후보 비교

**이 셀에서 하는 일**  
threshold를 바꾸면 metric이 어떻게 달라지는지 봅니다.

**확인할 점**  
threshold별 precision, recall, FP, FN을 비교합니다.

**왜 중요한가**  
threshold 변경은 개선 버튼이 아니라 오류 비용을 바꾸는 선택입니다.


In [ ]:
thresholds = [0.30, 0.40, 0.50, 0.60, 0.70]
threshold_rows: list[dict[str, object]] = []

for threshold in thresholds:
    threshold_frame = scored_dataframe.assign(
        prediction=lambda table, value=threshold: table["score"].ge(value).map(
            {True: positive_label, False: negative_label}
        )
    )
    row = metric_summary(threshold_frame, "vital_signs_test", threshold)
    threshold_rows.append(
        {
            "threshold": threshold,
            "predicted_high_risk": int((threshold_frame["prediction"] == positive_label).sum()),
            "predicted_high_risk_pct": round((threshold_frame["prediction"] == positive_label).mean() * 100, 2),
            "precision": row["precision"],
            "recall": row["recall"],
            "FP": row["FP"],
            "FN": row["FN"],
        }
    )

threshold_sweep = pd.DataFrame(threshold_rows)
display(threshold_sweep)


이 출력에서 확인할 핵심은 threshold가 오류 유형의 균형을 바꾼다는 점입니다. 낮은 threshold는 Recall을 높일 수 있지만 FP를 늘릴 수 있고, 높은 threshold는 FP를 줄일 수 있지만 FN을 늘릴 수 있습니다.

보고서에는 선택한 threshold와 함께 FP/FN trade-off를 씁니다. “threshold를 낮추면 Recall이 올라간다”에서 멈추지 않고, 오탐 증가가 운영 부담으로 이어지는지까지 적어야 합니다.

## 7. Evidence packet과 보고 문장

### 7-1. 모델 평가 결과를 QA handoff로 조립

마지막 정리는 앞의 출력들을 다음 notebook과 보고서로 넘길 수 있는 evidence packet으로 정리하는 것입니다. 이 packet은 최종 release 승인 문서가 아니라, 데이터 품질과 metric 연결 분석으로 넘기는 모델 평가 증거입니다.

`ready_for_metric_connection`은 데이터 품질 원인 후보를 더 확인할 준비가 되었다는 뜻입니다. 모델 성능이 충분하다는 승인 문구로 쓰면 안 됩니다.


### 따라하기 안내: QA 문장 정리

**이 셀에서 하는 일**  
앞에서 본 데이터 범위, metric, FP/FN, threshold 확인을 보고서 문장으로 묶습니다.

**확인할 점**  
결론이 지표 하나에만 기대지 않고, 어떤 데이터와 기준에서 나온 확인인지 포함하는지 봅니다.

**왜 중요한가**  
실무에서는 숫자만 제출하지 않습니다. 숫자를 만든 조건과 다음 확인 항목까지 함께 남겨야 재검토가 가능합니다.


In [ ]:
baseline_metric = metric_table.iloc[0].to_dict()
validation_delta = metric_delta.iloc[0].to_dict()
quality_delta = validation_quality_delta.iloc[0].to_dict()

if validation_delta.get("FP", 0) > 0 or validation_delta.get("FN", 0) > 0:
    next_action = "데이터 품질 신호와 FP/FN 변화를 연결해 원인 후보를 좁힙니다."
    decision = "ready_for_metric_connection"
else:
    next_action = "score와 prediction 분포를 확인해 threshold 민감도를 기록합니다."
    decision = "ready_for_metric_connection"

chapter_02_model_metric_packet = pd.DataFrame(
    [
        {
            "evidence": "test_data_brief",
            "observed": (
                f"source={test_source}, scope={execution_scope}, rows={len(test_dataframe)}, "
                f"feature_columns_present={int(feature_readiness['exists'].sum())}/{len(feature_columns)}"
            ),
            "qa_judgment": "metric 숫자보다 먼저 데이터 source, row grain, label/feature readiness를 고정했습니다.",
            "owner": "QA/Data Engineering",
            "next_action": "보고서에 Lite sample 여부와 데이터 범위를 함께 적습니다.",
        },
        {
            "evidence": "test_metric",
            "observed": (
                f"threshold={baseline_metric['threshold']}, "
                f"precision={baseline_metric['precision']}, recall={baseline_metric['recall']}, "
                f"FP={baseline_metric['FP']}, FN={baseline_metric['FN']}"
            ),
            "qa_judgment": "Accuracy 단독이 아니라 FP/FN과 Precision/Recall을 함께 해석합니다.",
            "owner": "QA",
            "next_action": "validation 비교와 threshold sweep을 함께 첨부합니다.",
        },
        {
            "evidence": "validation_data_quality_delta",
            "observed": (
                f"missing_row_pct_delta={quality_delta.get('missing_row_pct')}, "
                f"oxygen_saturation_min_delta={quality_delta.get('oxygen_saturation_min')}"
            ),
            "qa_judgment": "metric delta 전에 degraded 데이터의 입력 조건 차이를 확인했습니다.",
            "owner": "QA/Data Engineering",
            "next_action": "데이터 조건 변화가 score와 prediction 변화로 이어지는지 확인합니다.",
        },
        {
            "evidence": "validation_degradation_delta",
            "observed": (
                f"precision_delta={validation_delta.get('precision')}, "
                f"recall_delta={validation_delta.get('recall')}, "
                f"FP_delta={validation_delta.get('FP')}, FN_delta={validation_delta.get('FN')}"
            ),
            "qa_judgment": "같은 score 기준에서 데이터 조건 변화와 metric 변화가 함께 관측되는지 확인합니다.",
            "owner": "QA/Data Engineering",
            "next_action": next_action,
        },
        {
            "evidence": "threshold_sweep",
            "observed": f"checked_thresholds={thresholds}",
            "qa_judgment": "threshold 변경은 모델 개선이 아니라 FP/FN trade-off 검토입니다.",
            "owner": "QA/Product Owner",
            "next_action": "오탐/미탐 중 더 큰 운영 리스크를 release gate에서 명시합니다.",
        },
    ]
)

chapter_02_handoff = pd.DataFrame(
    [
        {
            "chapter": "02_model_quality",
            "decision": decision,
            "decision_reason": "data brief, score, prediction, confusion matrix, metric, threshold trade-off를 같은 기준에서 확인했습니다.",
            "open_candidates": "input_quality_shift, threshold_tradeoff, label_basis_review",
            "next_chapter_question": "데이터 품질 신호가 score/prediction/metric 변화와 어떻게 연결되는가?",
        }
    ]
)

report_sentence = (
    f"2장 모델 평가 결과, {execution_scope} 기준 test 데이터에서 source={test_source}, "
    f"rows={len(test_dataframe)}를 먼저 고정하고 threshold={baseline_metric['threshold']}를 적용해 "
    f"precision={baseline_metric['precision']}, recall={baseline_metric['recall']}, "
    f"FP={baseline_metric['FP']}, FN={baseline_metric['FN']}을 확인했습니다. "
    "따라서 Accuracy 단독 확인은 보류하고, validation 데이터 조건 차이와 threshold sweep을 함께 첨부해 "
    "데이터 조건 변화와 FP/FN trade-off를 다음 분석으로 넘깁니다."
)

print(report_sentence)


### 출력 확인: `chapter_02_model_metric_packet`

**확인할 점**  
accuracy, precision, recall, F1, AUROC/PR-AUC를 봅니다.

**왜 중요한가**  
precision은 FP의 영향을 받고, recall은 FN의 영향을 받습니다. accuracy 하나로는 오류 유형을 알 수 없습니다.

**다음 단계**  
confusion matrix의 FP/FN과 함께 해석합니다.

**주의할 점**  
metric에는 데이터셋과 threshold 조건을 같이 붙여야 합니다.


In [ ]:
display(chapter_02_model_metric_packet)


### 출력 확인: `chapter_02_handoff`

**확인할 점**  
앞 단계 결과가 어떤 evidence 묶음으로 정리됐는지 봅니다.

**왜 중요한가**  
packet과 handoff는 다음 장이나 최종 보고서로 넘길 요약입니다.

**다음 단계**  
남은 질문을 다음 장에서 계속 확인합니다.

**주의할 점**  
packet이 있다고 분석이 끝난 것은 아닙니다.


In [ ]:
display(chapter_02_handoff)


### 7-2. 실패 시 확인 포인트

실행이 실패하면 먼저 `package_module`, `data_source`, `execution_scope`, `feature_columns_present`를 확인합니다. CSV를 읽지 못해 embedded sample로 떨어졌다면 출력은 정상이어도 전체 데이터 기반 근거가 아니므로 보고서에는 fallback sample 기준이라고 적어야 합니다.

metric 값이 예상과 다르면 `raw data`, `label 표준화`, `feature readiness`, `score`, `threshold`, `prediction`, `label 비교`를 순서대로 확인합니다. 이 순서를 건너뛰면 label 문제, feature 결측, threshold 변경, 모델 출력 변화를 서로 혼동하게 됩니다.

validation 비교가 흔들릴 때는 모델 자체 결함으로 단정하지 않습니다. “degraded 데이터에서 feature 품질 차이가 먼저 관측되었고, 같은 score와 threshold 기준에서 FP/FN 변화가 함께 관측되었으므로 데이터 품질 신호와 threshold trade-off를 추가 확인합니다”처럼 현재 증거와 다음 action을 함께 씁니다.
